# VoxBind - Sample Metrics Dashboard

Summarises the cached `metrics.json` files written for every sampled pocket
under `voxbind/exps/<experiment>/samples/<run>/target_*/`.

### How to use
1. Run **section 0 (Setup)** once.
2. **Section 1** lists every experiment that has generated samples - review it.
3. In **section 2**, set `SELECTED` to an `idx` (or experiment name) from that list.
4. Run **sections 3-5** for the selected experiment; **section 6** compares all of them.

### Notes
- This dashboard only *reads* pre-computed `metrics.json` (it does not run
  RDKit or docking). Refresh those files with the Streamlit app
  (`notebook/webapp/app.py`).
- **Docking metrics (Vina) are optional.** They come from a separate evaluation
  step and may not be computed yet - targets without them show `NaN` in the
  `vina_*` columns and are flagged. Every other metric still prints normally.
- `status` column: `fresh` = metrics newer than `samples.sdf` -
  `stale` = out of date - `no-metrics` = not evaluated yet -
  `no-samples` = empty target directory.

In [11]:
# Section 0 - Setup: helper functions (run once).
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 100)


def show(obj, round_to=3):
    """Render a DataFrame/Series in Jupyter, falling back to plain text."""
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        obj = obj.round(round_to)
    try:
        from IPython.display import display
        display(obj)
    except Exception:
        print(obj)


# -- locate the repository ---------------------------------------------------
def find_repo_root() -> Path:
    """Walk up from the working directory until voxbind/exps/ is found."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "voxbind" / "exps").is_dir():
            return cand
    fallback = Path("/home/shpark/prj-denovo/VoxBind")
    if (fallback / "voxbind" / "exps").is_dir():
        return fallback
    raise FileNotFoundError("Could not find voxbind/exps/ - run from inside the repo.")


REPO = find_repo_root()
EXPS_ROOT = REPO / "voxbind" / "exps"
METRICS_VERSION = 2          # version stamp written by notebook/webapp/metrics.py
print(f"repo      : {REPO}")
print(f"exps root : {EXPS_ROOT}")


# -- experiment / run / target discovery -------------------------------------
def list_runs(exp_dir: Path) -> list:
    """Sampling-run subdirs under <exp>/samples/ that hold target_* folders."""
    samples = exp_dir / "samples"
    if not samples.is_dir():
        return []
    return sorted(d.name for d in samples.iterdir()
                  if d.is_dir() and any(d.glob("target_*")))


def list_experiments() -> list:
    """Experiment names with at least one non-empty sampling run."""
    if not EXPS_ROOT.is_dir():
        return []
    return sorted(p.name for p in EXPS_ROOT.iterdir()
                  if p.is_dir() and list_runs(p))


def list_targets(exp_name: str, run: str) -> list:
    """Sorted target_* directories for one (experiment, run)."""
    run_dir = EXPS_ROOT / exp_name / "samples" / run
    if not run_dir.is_dir():
        return []
    return sorted(t for t in run_dir.glob("target_*") if t.is_dir())


# -- metrics.json loading and freshness --------------------------------------
def metrics_status(target_dir: Path) -> str:
    """One of: fresh | stale | no-metrics | no-samples."""
    mp = target_dir / "metrics.json"
    sdf = target_dir / "samples.sdf"
    if not sdf.exists():
        return "no-samples"
    if not mp.exists():
        return "no-metrics"
    try:
        data = json.loads(mp.read_text())
    except (json.JSONDecodeError, OSError):
        return "no-metrics"
    if data.get("version") != METRICS_VERSION:
        return "stale"
    return "fresh" if mp.stat().st_mtime >= sdf.stat().st_mtime else "stale"


def load_metrics(target_dir: Path):
    """Parsed metrics.json dict, or None if absent / unreadable."""
    mp = target_dir / "metrics.json"
    if not mp.exists():
        return None
    try:
        return json.loads(mp.read_text())
    except (json.JSONDecodeError, OSError):
        return None


# -- docking metrics (Vina) - optional, may not be computed yet --------------
# Per-sample docking is expected either as a nested {"vina": {...}} dict
# (poc_evaluate.py convention) or as flat vina_* keys. When nothing is found,
# the vina_* columns stay NaN and the dashboard flags the target/experiment.
VINA_COLS = ["vina_score", "vina_min", "vina_dock"]
_VINA_NESTED = {"score_only": "vina_score", "minimize": "vina_min", "dock": "vina_dock"}


def sample_docking(sample: dict) -> dict:
    """Extract {vina_score, vina_min, vina_dock} from one sample record."""
    out = {}
    v = sample.get("vina")
    if isinstance(v, dict) and "error" not in v:
        for src, dst in _VINA_NESTED.items():
            if isinstance(v.get(src), (int, float)):
                out[dst] = float(v[src])
    for col in VINA_COLS:
        if isinstance(sample.get(col), (int, float)):
            out[col] = float(sample[col])
    return out


def target_docking(metrics: dict) -> dict:
    """Mean Vina metrics for a target - NaN for any not computed."""
    out = {c: float("nan") for c in VINA_COLS}
    acc = {c: [] for c in VINA_COLS}
    for s in metrics.get("samples", []):
        for c, val in sample_docking(s).items():
            acc[c].append(val)
    for c in VINA_COLS:
        if acc[c]:
            out[c] = float(np.mean(acc[c]))
    # fall back to aggregate-level keys, in case a future pipeline writes them there
    agg = metrics.get("aggregates", {})
    aliases = {
        "vina_score": ("vina_score_mean", "vina_score", "Vina score mean"),
        "vina_min":   ("vina_min_mean", "vina_min", "Vina min mean"),
        "vina_dock":  ("vina_dock_mean", "vina_dock", "Vina dock mean"),
    }
    for c, names in aliases.items():
        if math.isnan(out[c]):
            for n in names:
                if isinstance(agg.get(n), (int, float)):
                    out[c] = float(agg[n])
                    break
    return out


def has_docking(metrics) -> bool:
    """True if any Vina metric is present in this target's metrics."""
    return metrics is not None and any(
        not math.isnan(v) for v in target_docking(metrics).values()
    )


# -- table builders ----------------------------------------------------------
PER_TARGET_COLS = [
    "target", "status", "computed_at", "n_total", "n_valid",
    "validity", "uniqueness", "diversity",
    "qed", "sa", "logp", "lipinski", "clash_free",
] + VINA_COLS


def per_target_table(exp_name: str, run: str) -> pd.DataFrame:
    """One summary row per target_* directory of (experiment, run)."""
    rows = []
    for tdir in list_targets(exp_name, run):
        row = {"target": tdir.name, "status": metrics_status(tdir)}
        m = load_metrics(tdir)
        if m is not None:
            agg = m.get("aggregates", {})
            ca = m.get("computed_at", "") or ""
            row["computed_at"] = ca[:16].replace("T", " ")
            row["n_total"] = agg.get("n_total")
            row["n_valid"] = agg.get("n_valid")
            row["validity"] = agg.get("validity")
            row["uniqueness"] = agg.get("uniqueness")
            row["diversity"] = agg.get("diversity")
            row["qed"] = agg.get("qed_mean")
            row["sa"] = agg.get("sa_mean")
            row["logp"] = agg.get("logp_mean")
            row["lipinski"] = agg.get("lipinski_mean")
            samples = m.get("samples", [])
            if samples:
                row["clash_free"] = float(np.mean([
                    s.get("interactions", {}).get("n_clashes", 0) == 0
                    for s in samples
                ]))
            row.update(target_docking(m))
        rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    for c in PER_TARGET_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df[PER_TARGET_COLS]


def experiment_aggregate(df: pd.DataFrame) -> dict:
    """Mean of the per-target table over targets (NaN where not computed)."""
    numeric = ["validity", "uniqueness", "diversity", "qed", "sa", "logp",
               "lipinski", "clash_free"] + VINA_COLS
    out = {"n_targets": int(len(df))}
    out["evaluated"] = int((df["status"] == "fresh").sum()) if "status" in df else 0
    for c in numeric:
        out[c] = float(df[c].mean(skipna=True)) if (c in df and len(df)) else float("nan")
    return out


def per_sample_table(exp_name: str, run: str, target_name: str) -> pd.DataFrame:
    """Every generated molecule for one target (empty if not evaluated)."""
    tdir = EXPS_ROOT / exp_name / "samples" / run / target_name
    m = load_metrics(tdir)
    if m is None:
        return pd.DataFrame()
    rows = []
    for i, s in enumerate(m.get("samples", [])):
        ix = s.get("interactions", {})
        row = {
            "idx": i,
            "smiles": s.get("smiles", ""),
            "n_atoms": s.get("n_atoms"),
            "qed": s.get("qed"),
            "sa": s.get("sa"),
            "logp": s.get("logp"),
            "lipinski": s.get("lipinski"),
            "n_contacts": ix.get("n_contacts"),
            "n_clashes": ix.get("n_clashes"),
            "min_dist": ix.get("min_dist"),
        }
        for c in VINA_COLS:
            row[c] = np.nan
        row.update(sample_docking(s))
        rows.append(row)
    return pd.DataFrame(rows)


def first_evaluated(exp_name: str):
    """Return (run, target_name) of the fresh-metrics target with most samples.

    Falls back to the first run/target if none have fresh metrics.
    """
    runs = list_runs(EXPS_ROOT / exp_name)
    best = None  # (n_valid, run, target_name)
    for run in runs:
        for t in list_targets(exp_name, run):
            if metrics_status(t) != "fresh":
                continue
            m = load_metrics(t) or {}
            n = m.get("aggregates", {}).get("n_valid", 0) or 0
            if best is None or n > best[0]:
                best = (n, run, t.name)
    if best is not None:
        return best[1], best[2]
    if runs:
        ts = list_targets(exp_name, runs[0])
        return runs[0], (ts[0].name if ts else None)
    return None, None


print("helpers ready: list_experiments, list_runs, list_targets,")
print("               per_target_table, experiment_aggregate, per_sample_table")

repo      : /home/shpark/prj-denovo/VoxBind
exps root : /home/shpark/prj-denovo/VoxBind/voxbind/exps
helpers ready: list_experiments, list_runs, list_targets,
               per_target_table, experiment_aggregate, per_sample_table


## Experiment list

`EXPS` is auto-discovered: every experiment under `voxbind/exps/` that has at
least one non-empty sampling run. To curate which experiments the cross-
experiment comparison (section 6) reports on, replace it with a hand-picked
list, e.g. `EXPS = ["260514_voxbind_100ep340", "260515_voxbind_100ep_density"]`.

The `metrics` / `docking` columns are coverage counts (`computed / total`).

In [12]:
# Section 1 - list every experiment that has generated samples.
EXPS = list_experiments()

overview = []
for i, exp_name in enumerate(EXPS):
    runs = list_runs(EXPS_ROOT / exp_name)
    n_targets = n_fresh = n_dock = 0
    for run in runs:
        for tdir in list_targets(exp_name, run):
            n_targets += 1
            if metrics_status(tdir) == "fresh":
                n_fresh += 1
            if has_docking(load_metrics(tdir)):
                n_dock += 1
    overview.append({
        "idx": i,
        "experiment": exp_name,
        "runs": ", ".join(runs),
        "targets": n_targets,
        "metrics": f"{n_fresh}/{n_targets}",
        "docking": f"{n_dock}/{n_targets}" if n_dock else "not computed",
    })

if not EXPS:
    print(f"No experiments with generated samples under {EXPS_ROOT}")
else:
    print(f"{len(EXPS)} experiment(s) with generated samples:")
    show(pd.DataFrame(overview).set_index("idx"))


# Section 6 - compare every experiment in EXPS.
def target_reference(metrics):
    """Reference-ligand metrics for one target, or {} when absent.

    Read from the target's metrics.json `reference` key (qed/sa/logp/lipinski
    and the Vina scores). validity/uniqueness/diversity/clash_free are not
    defined for a single ligand.
    """
    ref = (metrics or {}).get("reference")
    if not isinstance(ref, dict) or "error" in ref:
        return {}
    out = {}
    for k in ("qed", "sa", "logp", "lipinski"):
        if isinstance(ref.get(k), (int, float)):
            out[k] = float(ref[k])
    v = ref.get("vina")
    if isinstance(v, dict) and "error" not in v:
        for src, dst in _VINA_NESTED.items():
            if isinstance(v.get(src), (int, float)):
                out[dst] = float(v[src])
    return out


def experiment_reference_row(exp_name):
    """Mean reference-ligand metrics over every target of an experiment.

    Gathered from each target's metrics.json - there is no `reference/` run on
    disk. Returned with the same keys as experiment_aggregate(); the set-level
    columns (validity/uniqueness/diversity/clash_free) stay NaN.
    """
    refs = []
    for run in list_runs(EXPS_ROOT / exp_name):
        for tdir in list_targets(exp_name, run):
            r = target_reference(load_metrics(tdir))
            if r:
                refs.append(r)
    metric_cols = ["qed", "sa", "logp", "lipinski"] + VINA_COLS
    row = {c: float("nan") for c in
           ["validity", "uniqueness", "diversity", "clash_free"] + metric_cols}
    row["n_targets"] = row["evaluated"] = len(refs)
    for c in metric_cols:
        vals = [r[c] for r in refs if c in r]
        if vals:
            row[c] = float(np.mean(vals))
    return row


def _highlight_best(df):
    """Comparison-table CSS: underline the best run within each experiment,
    bold the best across all experiments. Higher is better, except the
    lower-is-better Vina columns; logp and the count columns are not ranked.
    The 'reference' rows are a baseline and are excluded from the ranking.
    """
    higher = ["validity", "uniqueness", "diversity", "qed", "sa",
              "lipinski", "clash_free"]
    lower = ["vina_score", "vina_min", "vina_dock"]
    css = pd.DataFrame("", index=df.index, columns=df.columns)
    experiments = df.index.get_level_values("experiment")
    not_ref = pd.Series(df.index.get_level_values("run") != "reference",
                        index=df.index)
    for col in higher + lower:
        if col not in df.columns:
            continue
        series = df[col].where(not_ref)
        is_max = col in higher
        best = series.max() if is_max else series.min()
        if pd.notna(best):
            css.loc[series == best, col] += "font-weight: bold;"
        for exp in experiments.unique():
            grp = series[experiments == exp].dropna()
            if grp.empty:
                continue
            ebest = grp.max() if is_max else grp.min()
            in_exp = pd.Series(experiments == exp, index=df.index)
            css.loc[in_exp & (series == ebest), col] += "text-decoration: underline;"
    return css


def _color_vs_reference(df):
    """Comparison-table CSS: shade every non-reference run by how its metric
    compares with the reference ligand of the *same* experiment - light green
    when the run beats the reference, light red when it is worse. Columns with
    no reference value (validity/uniqueness/diversity/clash_free) or no defined
    direction (logp, the count columns) are left uncoloured.
    """
    higher = ["qed", "sa", "lipinski"]               # higher is better
    lower = ["vina_score", "vina_min", "vina_dock"]  # lower is better
    green = "background-color: #d8f0d8;"
    red = "background-color: #f5d8d8;"
    css = pd.DataFrame("", index=df.index, columns=df.columns)
    experiments = df.index.get_level_values("experiment")
    runs = df.index.get_level_values("run")
    for exp in experiments.unique():
        ref_rows = df[(experiments == exp) & (runs == "reference")]
        if ref_rows.empty:
            continue
        ref = ref_rows.iloc[0]
        run_idx = df.index[(experiments == exp) & (runs != "reference")]
        for col in higher + lower:
            if col not in df.columns or pd.isna(ref[col]):
                continue
            better_when_higher = col in higher
            for idx in run_idx:
                val = df.at[idx, col]
                if pd.isna(val):
                    continue
                better = val > ref[col] if better_when_higher else val < ref[col]
                css.at[idx, col] = green if better else red
    return css


cmp_rows = []
for exp_name in EXPS:
    for run in list_runs(EXPS_ROOT / exp_name):
        df = per_target_table(exp_name, run)
        if df.empty:
            continue
        cmp_rows.append({"experiment": exp_name, "run": run,
                         **experiment_aggregate(df)})
    # gather the reference ligands into a virtual 'reference' run (no folder)
    ref_row = experiment_reference_row(exp_name)
    if ref_row["n_targets"]:
        cmp_rows.append({"experiment": exp_name, "run": "reference", **ref_row})

cmp_df = pd.DataFrame(cmp_rows)
if cmp_df.empty:
    print("Nothing to compare - no experiments with samples.")
else:
    _cmp = cmp_df.set_index(["experiment", "run"])
    _styled = (_cmp.style
               .apply(_highlight_best, axis=None)
               .apply(_color_vs_reference, axis=None)
               .format(precision=3, na_rep="NaN"))
    show(_styled)
    print("  underline = best within an experiment  ·  bold = best across all")
    print("  green / red cell = run metric better / worse than its reference ligand")
    print("  'reference' run = mean reference-ligand metrics (baseline, not ranked)")
    missing = [c for c in VINA_COLS if cmp_df[c].isna().all()]
    if missing:
        print(f"  docking not computed anywhere - NaN columns: {', '.join(missing)}")

4 experiment(s) with generated samples:


,experiment,runs,targets,metrics,docking
idx,,,,,
0,260514_voxbind_100ep340,"res_ep7999_test10, res_ep7999_val",20,20/20,20/20
1,260515_voxbind_100ep_density,"res_ep7999_test10, res_ep7999_val",20,20/20,20/20
2,260516_voxbind_100ep_noise,"res_ep7999_test10, res_ep7999_val",20,20/20,19/20
3,reproduction,res,11,2/11,not computed


  underline = best within an experiment  ·  bold = best across all
  green / red cell = run metric better / worse than its reference ligand
  'reference' run = mean reference-ligand metrics (baseline, not ranked)


## Results

Set `SELECTED` below to an `idx` from the table in section 1 (or to an
experiment-name string), then run the remaining cells.

### Experiments - sample directories

In [13]:
# ======================================================================
#  vvv  SELECT THE EXPERIMENT  vvv   - idx from section 1, or its name
SELECTED = 1
# ======================================================================

if not EXPS:
    raise RuntimeError("No experiments to select - see section 1.")
if isinstance(SELECTED, int):
    sel_exp = EXPS[SELECTED]
elif SELECTED in EXPS:
    sel_exp = SELECTED
else:
    raise ValueError(f"SELECTED={SELECTED!r} is not a valid idx or name; "
                     f"choose from {EXPS}")

sel_runs = list_runs(EXPS_ROOT / sel_exp)
print(f"selected experiment : {sel_exp}")
print(f"sampling runs       : {sel_runs}")

agg_rows = []
for run in sel_runs:
    df = per_target_table(sel_exp, run)
    if df.empty:
        continue
    agg_rows.append({"run": run, **experiment_aggregate(df)})

if agg_rows:
    agg_df = pd.DataFrame(agg_rows).set_index("run")
    print(f"{sel_exp} - aggregate metrics (mean over targets)")
    show(agg_df)
    if agg_df[VINA_COLS].isna().all().all():
        print("  docking metrics (vina_*): not computed for this experiment")
else:
    print(f"{sel_exp}: no targets to aggregate.")

selected experiment : 260515_voxbind_100ep_density
sampling runs       : ['res_ep7999_test10', 'res_ep7999_val']
260515_voxbind_100ep_density - aggregate metrics (mean over targets)


,n_targets,evaluated,validity,uniqueness,diversity,qed,sa,logp,lipinski,clash_free,vina_score,vina_min,vina_dock
run,,,,,,,,,,,,,
res_ep7999_test10,10,10,0.99,1.0,0.622,0.503,0.536,2.651,4.638,0.99,-7.376,-8.114,-8.868
res_ep7999_val,10,10,1.00,1.0,0.721,0.548,0.589,1.892,4.790,1.00,-6.521,-7.326,-8.281


### Experiments - over runs

In [14]:
# Section 3 - per-target metrics for every run of the selected experiment.
for run in sel_runs:
    df = per_target_table(sel_exp, run)
    print(f"\n{'='*72}\n{sel_exp}  /  {run}\n{'='*72}")
    if df.empty:
        print("  (no target directories)")
        continue
    show(df.set_index("target"))
    n_no_dock = int(df[VINA_COLS].isna().all(axis=1).sum())
    if n_no_dock == len(df):
        print("  docking (vina_*): not computed for any target in this run")
    elif n_no_dock:
        print(f"  docking (vina_*): not computed for {n_no_dock}/{len(df)} targets")

if not sel_runs:
    print(f"{sel_exp}: no sampling runs.")


260515_voxbind_100ep_density  /  res_ep7999_test10


,status,computed_at,n_total,n_valid,validity,uniqueness,diversity,qed,sa,logp,lipinski,clash_free,vina_score,vina_min,vina_dock
target,,,,,,,,,,,,,,,
target_00,fresh,2026-05-18 14:56,10,9,0.9,1.0,0.799,0.425,0.573,0.158,4.778,1.0,-6.378,-6.725,-7.565
target_01,fresh,2026-05-18 14:56,10,10,1.0,1.0,0.608,0.530,0.483,4.501,4.500,1.0,-7.614,-8.298,-8.619
target_02,fresh,2026-05-18 14:57,10,10,1.0,1.0,0.577,0.618,0.544,3.277,4.800,1.0,-7.036,-7.370,-8.134
target_03,fresh,2026-05-18 14:57,10,10,1.0,1.0,0.698,0.492,0.583,3.529,4.500,0.9,-5.672,-6.729,-8.157
target_04,fresh,2026-05-18 14:58,10,10,1.0,1.0,0.491,0.429,0.490,3.027,4.300,1.0,-12.162,-12.487,-13.068
target_05,fresh,2026-05-18 14:58,10,10,1.0,1.0,0.765,0.569,0.615,1.087,5.000,1.0,-3.157,-3.993,-5.525
target_07,fresh,2026-05-18 14:59,10,10,1.0,1.0,0.629,0.543,0.561,3.389,4.600,1.0,-8.185,-8.969,-9.083
target_08,fresh,2026-05-18 06:31,10,10,1.0,1.0,0.529,0.331,0.449,3.095,4.200,1.0,-10.196,-10.760,-11.259
target_09,fresh,2026-05-18 06:31,10,10,1.0,1.0,0.503,0.630,0.507,2.560,4.900,1.0,-4.998,-7.132,-8.004



260515_voxbind_100ep_density  /  res_ep7999_val


,status,computed_at,n_total,n_valid,validity,uniqueness,diversity,qed,sa,logp,lipinski,clash_free,vina_score,vina_min,vina_dock
target,,,,,,,,,,,,,,,
target_00,fresh,2026-05-18 15:01,10,10,1.0,1.0,0.793,0.485,0.582,1.113,5.0,1.0,-1.984,-4.550,-7.020
target_01,fresh,2026-05-18 15:01,10,10,1.0,1.0,0.669,0.590,0.631,1.176,5.0,1.0,-7.402,-7.889,-8.248
target_02,fresh,2026-05-18 15:01,10,10,1.0,1.0,0.788,0.450,0.606,3.576,4.6,1.0,-6.872,-7.434,-8.738
target_03,fresh,2026-05-18 15:02,10,10,1.0,1.0,0.560,0.485,0.508,1.651,4.8,1.0,-7.696,-8.124,-9.029
target_04,fresh,2026-05-18 15:02,10,10,1.0,1.0,0.799,0.612,0.615,0.833,5.0,1.0,-6.232,-7.048,-7.450
target_05,fresh,2026-05-18 15:03,10,10,1.0,1.0,0.838,0.663,0.682,2.576,5.0,1.0,-6.035,-6.513,-7.469
target_06,fresh,2026-05-18 15:03,10,10,1.0,1.0,0.787,0.521,0.598,2.009,4.8,1.0,-6.750,-7.177,-8.103
target_07,fresh,2026-05-18 15:06,10,10,1.0,1.0,0.546,0.286,0.471,3.037,3.7,1.0,-8.643,-9.408,-10.326
target_08,fresh,2026-05-18 15:06,9,9,1.0,1.0,0.843,0.649,0.657,1.737,5.0,1.0,-5.705,-6.486,-7.191


### Experiments - per sample

Every generated molecule for a single target. Leave `RUN` / `SELECTED_TARGET`
as `None` to auto-pick the evaluated target with the most samples, or set them
to an index or name to inspect a specific one.

In [15]:
# Section 5 - per-sample metrics for one target.
RUN = None              # None -> auto | index into sel_runs | run-name string
SELECTED_TARGET = None  # None -> auto | index into the run's targets | target name

if not sel_runs:
    print(f"{sel_exp}: no sampling runs.")
else:
    if RUN is None and SELECTED_TARGET is None:
        run, tname = first_evaluated(sel_exp)
    else:
        run = sel_runs[RUN] if isinstance(RUN, int) else (RUN or sel_runs[0])
        tnames = [t.name for t in list_targets(sel_exp, run)]
        if SELECTED_TARGET is None:
            tname = tnames[0] if tnames else None
        elif isinstance(SELECTED_TARGET, int):
            tname = tnames[SELECTED_TARGET]
        else:
            tname = SELECTED_TARGET

    if run is None or tname is None:
        print(f"{sel_exp}: no samples to display.")
    else:
        tdir = EXPS_ROOT / sel_exp / "samples" / run / tname
        df = per_sample_table(sel_exp, run, tname)
        print(f"{sel_exp}  /  {run}  /  {tname}   "
              f"(status: {metrics_status(tdir)})")
        if df.empty:
            print("  no metrics.json - this target has not been evaluated yet")
        else:
            show(df.set_index("idx"))
            if df[VINA_COLS].isna().all().all():
                print("  docking (vina_*): not computed for these samples")

260515_voxbind_100ep_density  /  res_ep7999_test10  /  target_01   (status: fresh)


,smiles,n_atoms,qed,sa,logp,lipinski,n_contacts,n_clashes,min_dist,vina_score,vina_min,vina_dock
idx,,,,,,,,,,,,
0,CCC[C@@](C)(O)C(=O)CCCCN=C1C=C(CC2=CCC(C)=C[C@@H]2C)CC=N1,28,0.406,0.57,5.379,4,18,0,2.940,-5.117,-5.703,-7.047
1,CC[C@@H]1C=C2C(O)=CCC3=C2[C@@H]2C(=CC=CC21)C[C@@H]3OCCO[C@@H]1CC=CN2CCC[C@@H](O)[C@H]2C1,35,0.520,0.48,5.130,4,26,0,2.917,-7.247,-8.210,-8.830
2,CN=C1C=C2C3C(C1)N(O)C=C[C@H]3[C@@H]1C([C@H]3CCCCC3=O)COC3=CCC[C@@H]2[C@@H]31,30,0.708,0.47,4.152,5,20,0,2.912,-9.443,-9.750,-9.817
3,CCC(CC)NC[C@@H]1C[C@@]2(CC[C@@H]3[C@H]4NCC5=CCC6=C(C7[C@H](CNC[C@@H]7[C@@H]32)C(O)=C6)[C@@H]54)[...,35,0.390,0.39,3.663,5,25,0,2.903,-7.173,-7.937,-8.415
4,C[C@@H]1CNO[C@H]2c3cccc(c3)COC(CO)=CCCC=C[C@@]3(CC[C@H](C3)[C@H](O)c3cccc(c3)[C@H]2O)[C@H]1O,39,0.341,0.28,4.556,4,20,0,2.824,-10.265,-10.591,-10.660
5,CC[C@@H](N=C1CCC2=C3N1c1c(C4COCOC4)cccc1[C@H]1CN=C(CC2)N31)OC,31,0.736,0.57,3.930,5,19,0,3.086,-7.635,-7.643,-7.041
6,C=C1Cc2cccc3c2N(C1)[C@H]1C[C@@H](CCC2=CC(C)=C[C@@H](C[C@H](C)O)[C@@H]2C)COC1=C3,32,0.576,0.53,6.055,4,24,0,2.914,-7.717,-9.253,-9.276
7,CCC1=NC=C(OC)C2=CC[C@H]3C(=O)C[C@@H](CO[C@@]4(O)OCCN4c4ccccc4)CNN3[C@H]21,34,0.612,0.55,1.959,5,21,0,3.197,-8.076,-8.095,-8.372
8,C=C[C@@H]([C@@H](C)[C@@H](C)[C@H]1CCC[C@H]2C(C)C[C@H]3C(C=O)=CC(=O)N=C3[C@@H]12)[C@H](C)CC,29,0.396,0.50,5.902,4,18,0,2.900,-7.489,-7.768,-8.030
